# SAM Instance Debugger

Build the cache first with `build_rgb_cache.py`, then use this notebook to step the standalone SAM-instance algorithm frame by frame.

The notebook intentionally keeps the control surface small:
- adjust the hyperparameters in one cell
- recreate the debugger after edits
- use the widget to step / seek / jump to the next seed frame
- inspect detailed decisions and bucket state in the text panel

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '6'

In [ ]:
from pathlib import Path

from map_runtime.sam_instance_debug import CachedSAMInstanceDebugger, create_debugger_widget
from map_runtime.sam_instance_runtime import SAMInstancePipelineConfig, SAMInstanceRuntimeConfig
from map_runtime.sam_masks import SAMMaskExtractorConfig
from map_runtime.sam2_tracking import SAMTrackerConfig

In [ ]:
# ---------------------------------------------------------------------------------->
# 'metrics': {'instance': {'ap_25': 0.49920062918979147,
#    'ap_50': 0.14419289419289422,
#    'ap_55': 0.14419289419289422,
#    'ap_60': 0.11072598572598571,
#    'ap_65': 0.0908954408954409,
#    'ap_70': 0.059939159939159936,
#    'ap_75': 0.00974025974025974,
#    'ap_80': 0.00974025974025974,
#    'ap_85': 0.0010822510822510823,
#    'ap_90': 0.0010822510822510823,
#    'ap_95': 0.0010822510822510823,
#    'ap': 0.057267364767364784}},

# 'metrics': {'instance': {'ap_25': 0.8727133081971792,
#    'ap_50': 0.7212121212121212,
#    'ap_55': 0.5836124401913876,
#    'ap_60': 0.5055561251613884,
#    'ap_65': 0.5055561251613884,
#    'ap_70': 0.4143324219640009,
#    'ap_75': 0.20426217057796003,
#    'ap_80': 0.13636363636363635,
#    'ap_85': 0.09595959595959595,
#    'ap_90': 0.02777777777777778,
#    'ap_95': 0.017676767676767676,
#    'ap': 0.3212309182046024}},
# ---------------------------------------------------------------------------------->

CACHE_DIR = Path("data/output/rgb_caches/ScanNet/scene0011_00")

# seed_mask_config = SAMMaskExtractorConfig(
#     model_level=13,
#     sort_mode="area",
#     min_mask_area_perc=0.01,
#     points_per_side=24,
#     points_per_batch=128,
#     pred_iou_thresh=0.88,
#     stability_score_thresh=0.92,
#     stability_score_offset=1.0,
#     mask_threshold=0.0,
#     box_nms_thresh=0.7,
#     crop_n_layers=0,
#     crop_nms_thresh=0.7,
#     crop_overlap_ratio=0.6,
#     crop_n_points_downscale_factor=1,
#     point_grids=None,
#     min_mask_region_area=0,
#     output_mode="binary_mask",
#     use_m2m=False,
#     multimask_output=True,
#     score_pred_iou_power=2.0,
#     score_stability_power=1.0,
#     score_area_power=0.0,
#     mask_overlap_rescore_thresh=0.0,
#     mask_overlap_rescore_power=1.0,
#     mask_dedupe_iou_thresh=0.85,
#     mask_containment_thresh=0.0,
# )
# tracker_config = SAMTrackerConfig(
#     model_level=24,
#     max_num_objects=16,
# )
# pipeline_config = SAMInstancePipelineConfig(
#     point_gid_slots=10,
#     reuse_inside_frac_th=0.40,
#     reuse_outside_frac_th=0.10,
#     min_mask_points=1,
#     min_track_visible_points=1,
#     prune_every_frames=64,  # 50
#     prune_stale_gap_frames=100000,  # 2000
#     prune_min_support_ratio=0.0,  # 0.0005
#     prune_min_points=1000,  # 5000
# )

# config = SAMInstanceRuntimeConfig(
#     seed_mask=seed_mask_config,
#     tracker=tracker_config,
#     pipeline=pipeline_config,
# )

config = SAMInstanceRuntimeConfig()

DEVICE = "cuda"

In [ ]:
debugger = CachedSAMInstanceDebugger(CACHE_DIR, config, device=DEVICE)
create_debugger_widget(debugger)

In [ ]:
debugger.show_sam_masks()

In [ ]:
debugger.show(gid=20)

In [ ]:
# Bucket / point-membership queries on the current debugger state.
# Run debugger.step(), debugger.seek(...), or use the widget first so debugger.current_view is populated.
#
# gid = 3
# bucket = debugger.buckets[gid]
# bucket
#
# point_id = 12345
# debugger.point_gids[point_id]  # K=10 gid slots for one global 3D point
#
# row, col = 200, 300
# point_id = int(debugger.current_view["point_ids_after"][row, col])
# if point_id >= 0:
#     print("pixel -> point_id", point_id)
#     print("gid slots", debugger.point_gids[point_id])
#
# active_buckets = {gid: debugger.buckets[gid] for gid in debugger.current_view["seeded_gids"]}
# active_buckets


In [ ]:
# Manual inspection examples
# debugger.step()
# debugger.seek(80)
# print(debugger.current_text_summary())
# debugger.current_view["decisions"]
# debugger.show(local_id=3)  # seed frames only
# debugger.show(gid=3)       # current projected global-instance support for gid=3
# debugger.show(local_id=3, gid=3)  # compare seed local mask vs current projected gid support
# debugger.simulate_sam_onlyseed_video(upto=100, map_every=1)
# debugger.simulate_video(upto=-1)

In [ ]:
debugger.simulate_sam_onlyseed_video(upto=200, map_every=1)

In [ ]:
debugger.get_metrics(scannet_raw_root="/robodata/smodak/datasets/scannet_v2/scans", min_component_size=2000, ovo_score_th=0.0, chunk_size=100_000, use_optimal_text_matching=True)

In [ ]:
debugger.get_metrics(scannet_raw_root="/robodata/smodak/datasets/scannet_v2/scans", min_component_size=2000, ovo_score_th=0.0, chunk_size=100_000, use_optimal_text_matching=True, use_optimal_collapse=True)